In [3]:
# cell 1: Import required libraries for data manipulation and flight schedules
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("Libraries imported successfully! Ready for the next step.")

Pandas version: 3.0.1
Libraries imported successfully! Ready for the next step.


In [19]:
# cell 2: Import the dataset using the clean filename and encoding fix
file_path = "../data/British Airways Summer Schedule Dataset - Forage Data Science Task 1.xlsx"
df = pd.read_excel(file_path)

In [20]:
df.head()

,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX
0,2025-09-02,14:19:00,Afternoon,BA,BA5211,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,10,38
1,2025-06-10,06:42:00,Morning,BA,BA7282,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,7,28
2,2025-10-27,15:33:00,Afternoon,BA,BA1896,LHR,FRA,Germany,Europe,SHORT,A320,0,17,163,0,11,40
3,2025-06-15,18:29:00,Evening,BA,BA5497,LHR,IST,Turkey,Europe,SHORT,A320,0,8,172,0,16,54
4,2025-08-25,20:35:00,Evening,BA,BA1493,LHR,FRA,Germany,Europe,SHORT,A320,0,13,167,0,6,27


In [21]:
# Inspect the precise column data types and missing value counts
print("--- Dataset Structural Info ---")
df.info()

# Clean up column names right away by removing any hidden spaces
df.columns = df.columns.str.strip()

--- Dataset Structural Info ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   FLIGHT_DATE           10000 non-null  datetime64[us]
 1   FLIGHT_TIME           10000 non-null  object        
 2   TIME_OF_DAY           10000 non-null  str           
 3   AIRLINE_CD            10000 non-null  str           
 4   FLIGHT_NO             10000 non-null  str           
 5   DEPARTURE_STATION_CD  10000 non-null  str           
 6   ARRIVAL_STATION_CD    10000 non-null  str           
 7   ARRIVAL_COUNTRY       10000 non-null  str           
 8   ARRIVAL_REGION        10000 non-null  str           
 9   HAUL                  10000 non-null  str           
 10  AIRCRAFT_TYPE         10000 non-null  str           
 11  FIRST_CLASS_SEATS     10000 non-null  int64         
 12  BUSINESS_CLASS_SEATS  10000 non-null  int64         
 

In [22]:
# cell 5: Filter the data for a representative sample matching your AM groupings
# We filter for flights departing in the 'Morning' and bound for 'Europe' (Short/Mid-haul)
am_flight_sample = df[(df['TIME_OF_DAY'] == 'Morning') & (df['ARRIVAL_REGION'] == 'Europe')].copy()

print(f"Sample created successfully! Found {len(am_flight_sample)} morning European flights.")
print("\n--- Breakdown of HAUL types in our sample ---")
print(am_flight_sample['HAUL'].value_counts())

Sample created successfully! Found 2069 morning European flights.

--- Breakdown of HAUL types in our sample ---
HAUL
SHORT    2069
Name: count, dtype: int64


In [23]:
# cell 6: Split the dataset sample into Short-haul AM and Mid-haul AM based on destination
def determine_haul_subgroup(station):
    # Standard BA Terminal 3 longer European routes (typically over 3.5 hours)
    mid_haul_stations = ['LCA', 'IST', 'ATH'] 
    if station in mid_haul_stations:
        return 'Mid-haul AM'
    else:
        return 'Short-haul AM'

# Apply the classification function to our representative morning sample
am_flight_sample['FLIGHT_CATEGORY'] = am_flight_sample['ARRIVAL_STATION_CD'].apply(determine_haul_subgroup)

print("--- Breakdown of the calculated Flight Categories in our sample ---")
print(am_flight_sample['FLIGHT_CATEGORY'].value_counts())


--- Breakdown of the calculated Flight Categories in our sample ---
FLIGHT_CATEGORY
Short-haul AM    1851
Mid-haul AM       218
Name: count, dtype: int64


In [24]:
# cell 7: Calculate total capacity and simulate expected lounge demand based on lookup logic
# 1. Calculate the total available seats for every flight row
am_flight_sample['TOTAL_SEATS'] = (
    am_flight_sample['FIRST_CLASS_SEATS'] + 
    am_flight_sample['BUSINESS_CLASS_SEATS'] + 
    am_flight_sample['ECONOMY_SEATS']
)

# 2. Define the lookup rules mapping using an optimized dictionary mapping
tier_rates = {
    'Short-haul AM': {'T1': 0.05, 'T2': 0.20, 'T3': 0.60},
    'Mid-haul AM':   {'T1': 0.07, 'T2': 0.25, 'T3': 0.65}
}

# 3. Compute simulated customer lounge traffic volumes across all three tiers
am_flight_sample['CALCULATED_TIER1_PAX'] = am_flight_sample.apply(lambda r: round(r['TOTAL_SEATS'] * tier_rates[r['FLIGHT_CATEGORY']]['T1'], 1), axis=1)
am_flight_sample['CALCULATED_TIER2_PAX'] = am_flight_sample.apply(lambda r: round(r['TOTAL_SEATS'] * tier_rates[r['FLIGHT_CATEGORY']]['T2'], 1), axis=1)
am_flight_sample['CALCULATED_TIER3_PAX'] = am_flight_sample.apply(lambda r: round(r['TOTAL_SEATS'] * tier_rates[r['FLIGHT_CATEGORY']]['T3'], 1), axis=1)

print("✅ Mathematical models applied to all 2,069 flights successfully!")
print("\n--- Model Evaluation: Calculated Lounge Projections vs Dataset Logged Pax ---")
# Show a summary slice of your columns to verify our logic outputs
display(am_flight_sample[['FLIGHT_NO', 'ARRIVAL_STATION_CD', 'FLIGHT_CATEGORY', 'TOTAL_SEATS', 'CALCULATED_TIER1_PAX', 'CALCULATED_TIER2_PAX', 'CALCULATED_TIER3_PAX']].head(3))


✅ Mathematical models applied to all 2,069 flights successfully!

--- Model Evaluation: Calculated Lounge Projections vs Dataset Logged Pax ---


,FLIGHT_NO,ARRIVAL_STATION_CD,FLIGHT_CATEGORY,TOTAL_SEATS,CALCULATED_TIER1_PAX,CALCULATED_TIER2_PAX,CALCULATED_TIER3_PAX
9,BA4878,IST,Mid-haul AM,180,12.6,45.0,117.0
10,BA8924,AMS,Short-haul AM,180,9.0,36.0,108.0
12,BA8213,MUC,Short-haul AM,180,9.0,36.0,108.0


In [25]:
# cell 8: Calculate overall total lounge traffic metrics across the sample
total_flights = len(am_flight_sample)
total_seats_deployed = am_flight_sample['TOTAL_SEATS'].sum()

total_t1 = int(am_flight_sample['CALCULATED_TIER1_PAX'].sum())
total_t2 = int(am_flight_sample['CALCULATED_TIER2_PAX'].sum())
total_t3 = int(am_flight_sample['CALCULATED_TIER3_PAX'].sum())
grand_total_lounge_users = total_t1 + total_t2 + total_t3

print("====================================================")
print("   BRITISH AIRWAYS TERMINAL 3 LOUNGE CAPACITY LOG   ")
print("====================================================")
print(f"Total AM Flights Modeled:            {total_flights:,}")
print(f"Total Airline Seat Capacity:         {total_seats_deployed:,}")
print("----------------------------------------------------")
print(f"Projected T1 (Concorde) Pax Volume:  {total_t1:,} passengers")
print(f"Projected T2 (First) Pax Volume:     {total_t2:,} passengers")
print(f"Projected T3 (Club) Pax Volume:      {total_t3:,} passengers")
print("----------------------------------------------------")
print(f"TOTAL PREDICTED LOUNGE TRAFFIC:      {grand_total_lounge_users:,} passengers")
print("====================================================")


   BRITISH AIRWAYS TERMINAL 3 LOUNGE CAPACITY LOG   
Total AM Flights Modeled:            2,069
Total Airline Seat Capacity:         372,420
----------------------------------------------------
Projected T1 (Concorde) Pax Volume:  19,405 passengers
Projected T2 (First) Pax Volume:     76,446 passengers
Projected T3 (Club) Pax Volume:      225,414 passengers
----------------------------------------------------
TOTAL PREDICTED LOUNGE TRAFFIC:      321,265 passengers
